In [1]:
from discovery_child_development.getters import gtr
import pandas as pd

import datetime

import importlib
importlib.reload(gtr);

from discovery_child_development import PROJECT_DIR
ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

2024-06-17 10:48:11,791 - botocore.credentials - INFO - Found credentials in environment variables.
2024-06-17 10:48:13,233 - datasets - INFO - PyTorch version 2.1.2 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
projects_df = gtr.get_gtr_from_s3(table='projects')

In [3]:
funds_dict = gtr.get_gtr_from_s3(table='funds')

In [5]:
funds_id = []
start = []
ends = []
value = []
currencies = []

for funds in funds_dict:
    funds_id.append(funds['id'])
    start.append(funds['start'])
    ends.append(funds['end'])
    value.append(funds['valuePounds']['amount'] if 'valuePounds' in funds else None)
    currencies.append(funds['valuePounds']['currencyCode'] if 'valuePounds' in funds else None)

funds_df = (
    pd.DataFrame({'funds_id': funds_id, 'start': start, 'ends': ends, 'amount': value, 'currency': currencies})
    .assign(start = lambda df: df.start.apply(lambda x: datetime.datetime.fromtimestamp(int(x)/1000).strftime('%Y-%m-%d')))
    .assign(ends = lambda df: df.ends.apply(lambda x: datetime.datetime.fromtimestamp(int(x)/1000).strftime('%Y-%m-%d')))
)


In [6]:
funds_df.head(1)

,funds_id,start,ends,amount,currency
0,2293906D-CC22-4E09-A91E-CE2CC6FE6CCD,2022-08-01,2026-07-31,1529795,GBP


In [7]:
projects_df[0].keys()

dict_keys(['links', 'id', 'href', 'created', 'identifiers', 'title', 'status', 'grantCategory', 'leadFunder', 'leadOrganisationDepartment', 'abstractText', 'techAbstractText', 'healthCategories', 'researchActivities', 'researchSubjects', 'researchTopics', 'rcukProgrammes'])

In [14]:
projects_df[6]

{'links': {'link': [{'href': 'http://internal-gtr-tomcat-alb-611010599.eu-west-2.elb.amazonaws.com:8080/gtr/api/persons/D15B119B-068D-4F6A-9B32-BCB4F63FBAF5',
    'rel': 'SUPER_PER',
    'otherAttributes': {}},
   {'href': 'http://internal-gtr-tomcat-alb-611010599.eu-west-2.elb.amazonaws.com:8080/gtr/api/persons/DEA634BD-85CB-4FF4-A1A9-43735D7048B1',
    'rel': 'SUPER_PER',
    'otherAttributes': {}},
   {'href': 'http://internal-gtr-tomcat-alb-611010599.eu-west-2.elb.amazonaws.com:8080/gtr/api/organisations/09D41C19-6E04-4937-902C-C3CD52E0683F',
    'rel': 'LEAD_ORG',
    'otherAttributes': {}},
   {'href': 'http://internal-gtr-tomcat-alb-611010599.eu-west-2.elb.amazonaws.com:8080/gtr/api/funds/3D9563CC-9AA4-45BE-99C9-B940FEA9E9AD',
    'rel': 'FUND',
    'start': 1569888000000,
    'end': 1696032000000,
    'otherAttributes': {}},
   {'href': 'http://internal-gtr-tomcat-alb-611010599.eu-west-2.elb.amazonaws.com:8080/gtr/api/projects/1BBC848E-8516-4275-90EC-9965A01561A7',
    'rel': '

In [15]:
ids = []
titles = []
abstracts = []
tech_abstracts = []
starts = []
ends = []
funds_id = []
leadFunder = []
identifiers = []

for project in projects_df:
    ids.append(project['id'])
    titles.append(project['title']) if 'title' in project.keys() else titles.append('')
    abstracts.append(project['abstractText']) if 'abstractText' in project.keys() else abstracts.append('')
    tech_abstracts.append(project['techAbstractText']) if 'techAbstractText' in project.keys() else tech_abstracts.append('')

    funds = [l for l in project['links']['link'] if l['rel'] == "FUND"][0]
    starts.append(datetime.datetime.fromtimestamp(int(funds['start'])/1000).strftime('%Y-%m-%d'))
    ends.append(datetime.datetime.fromtimestamp(int(funds['end'])/1000).strftime('%Y-%m-%d'))
    funds_id.append(funds['href'].split('/')[-1])
    leadFunder.append(project['leadFunder'])
    identifiers.append(project['identifiers']['identifier'][0]['value'])

gtr_texts_df = (
    pd.DataFrame({
        'id': ids,
        'title': titles,
        'abstract': abstracts,
        'tech_abstract': tech_abstracts,
        'start': starts,
        "end": ends,
        "funds_id": funds_id,
        "leadFunder": leadFunder,
        "identifiers": identifiers
    })
    .assign(text = lambda x: x['title'] + '. ' + x['abstract'] + '. ' + x['tech_abstract'])
    .merge(funds_df[['funds_id', 'amount', 'currency']], on='funds_id', how='left', suffixes=('', '_funds'))
)


In [19]:
gtr_texts_df.sample(1)

,id,title,abstract,tech_abstract,start,end,funds_id,leadFunder,identifiers,text,amount,currency
70167,03AD2BF0-8613-4306-B110-86943DDD1409,Geographical Information Science (GIS). Master...,Doctoral Training Partnerships: a range of pos...,,2009-09-01,2011-08-31,DA0B082D-7F02-4C8E-9B4D-DE8631F7E40A,NERC,NE/H525697/1,Geographical Information Science (GIS). Master...,123038,GBP


In [20]:
gtr_texts_df.drop(['abstract', 'tech_abstract'], axis=1).query("start >= '2013-01-01'").to_csv(ENRICHED_DATA_DIR / 'gtr_texts.csv', index=False)

## Check relevant gtr texts

In [88]:
from nesta_ds_utils.loading_saving import S3
from discovery_child_development import S3_BUCKET

In [89]:
gtr_texts_relevant_df = S3.download_obj(S3_BUCKET, f"data/outputs/binary_classifier/gtr_texts_relevance_labelled.csv", "dataframe")

In [94]:
gtr_texts_relevant_df.head(1)

,Unnamed: 0,id,start,end,text,predictions
0,0,9FCC3406-8050-4D4B-A4D6-14ABC3A9AC51,2022-01-01,2024-12-31,"20-BBSRC/NSF-BIO Regulatory control of innate immune response in marine invertebrates. Animals live in constantly changing environments that are rich in microbial life. Immune systems orchestrate the complex, dynamic relationships between animals and microbes by both eliminating pathogenic microbes as well as promoting the growth of beneficial microbiota. However, our understanding of innate immune mechanisms in most animal phyla, including vertebrates, remains limited. The proposed research addresses this considerable knowledge gap by exploiting the experimental advantages of echinoderm larvae to investigate immune responses in a relatively unexplored animal lineage. Echinoderm larvae are morphologically simple, free-swimming multicellular organisms equipped with a sophisticated cellular and molecular immune system. By integrating recently available sequencing data from several echinoderm species with technology for measuring gene expression at single-cell resolution, the proposed...",0


In [93]:
pd.set_option("max_colwidth", 1000)
gtr_texts_relevant_df.query("predictions == 1").to_csv(ENRICHED_DATA_DIR / 'gtr_texts_relevant.csv', index=False)